In [ ]:
!pip -q uninstall -y dask distributed
!pip -q install -q "dask[distributed]==2023.12.1" "distributed==2023.12.1"
!pip -q install -q --upgrade arboreto


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.0/999.0 kB 73.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rapids-dask-dependency 25.10.0 requires dask==2025.9.1, but you have dask 2023.12.1 which is incompatible.
rapids-dask-dependency 25.10.0 requires distributed==2025.9.1, but you have distributed 2023.12.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.9 MB/s eta 0:00:00


In [ ]:
%%bash
set -e
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
./bin/micromamba --version


bin/micromamba
2.5.0


In [ ]:
%%bash
set -e

# Create env "scenic" with Python 3.11 and core deps
./bin/micromamba create -y -n scenic -c conda-forge \
  python=3.11 pandas scikit-learn numpy scipy \
  dask=2024.2.1 distributed=2024.2.1 \
  pip

# Install arboreto via pip inside the env
./bin/micromamba run -n scenic pip -q install arboreto
./bin/micromamba run -n scenic python -c "import pandas as pd; import dask; import distributed; print('pandas',pd.__version__,'dask',dask.__version__,'distributed',distributed.__version__)"




Transaction

  Prefix: /root/.local/share/mamba/envs/scenic

  Updating specs:

   - python=3.11
   - pandas
   - scikit-learn
   - numpy
   - scipy
   - dask=2024.2.1
   - distributed=2024.2.1
   - pip


  Package                                  Version  Build                 Channel          Size
─────────────────────────────────────────────────────────────────────────────────────────────────
  Install:
─────────────────────────────────────────────────────────────────────────────────────────────────

  + _libgcc_mutex                              0.1  conda_forge           conda-forge       3kB
  + _openmp_mutex                              4.5  2_gnu                 conda-forge      24kB
  + aws-c-auth                               0.9.3  hef928c7_0            conda-forge     133kB
  + aws-c-cal                               0.9.13  h2c9d079_1            conda-forge      56kB
  + aws-c-common                            0.12.6  hb03c661_0            conda-forge     240kB
  + aws-c

In [ ]:
%%bash
set -e
wget -q -O /content/allTFs_hg38.txt https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt
wc -l /content/allTFs_hg38.txt | cat


1892 /content/allTFs_hg38.txt


In [ ]:
!pip install scanpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 85.9 MB/s eta 0:00:00


In [ ]:
!pip install scikit-misc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 18.4 MB/s eta 0:00:00


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

in_path = "/content/Combined_UMI_table.txt"
tf_path = "/content/allTFs_hg38.txt"
targets = ["PRM1", "PRM2"]


df = pd.read_csv(in_path, sep="\t", index_col=0)
adata = sc.AnnData(df.T)

adata.var_names = adata.var_names.astype(str).str.strip()
adata.var_names_make_unique()


adata.layers["counts"] = adata.X.copy()

# QC metrics
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

# Filter cells
min_counts = 1000
min_genes = 200
max_pct_mt = 15

adata = adata[
    (adata.obs["total_counts"] >= min_counts)
    & (adata.obs["n_genes_by_counts"] >= min_genes)
    & (adata.obs["pct_counts_mt"] <= max_pct_mt),
    :
].copy()

# Filter genes
sc.pp.filter_genes(adata, min_cells=10)

# Ensure targets exist after QC
missing_targets = [g for g in targets if g not in adata.var_names]
if missing_targets:
    raise ValueError(f"Targets missing after QC/gene filtering: {missing_targets}")

# Normalize + log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# HVG selection
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    layer="counts",
    subset=False
)

panel = set(adata.var_names[adata.var["highly_variable"]]) | set(["PRM1", "PRM2"])
panel = [g for g in panel if g in adata.var_names]
adata = adata[:, panel].copy()

X = pd.DataFrame(
    adata.X.A if hasattr(adata.X, "A") else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

# z-score
X = (X - X.mean(axis=0)) / X.std(axis=0).replace(0, np.nan)
X = X.fillna(0.0)

expr = X

tf_list = (
    pd.read_csv(tf_path, header=None)[0]
    .astype(str).str.strip().tolist()
)

# TFs present after HVG/panel filtering
tfs_in_data = [t for t in tf_list if t in expr.columns]
print(f"TFs found in dataset after HVG/panel filtering: {len(tfs_in_data)}")

# Final matrix = TFs ∪ targets
missing_targets2 = [g for g in targets if g not in expr.columns]
if missing_targets2:
    raise ValueError(f"After HVG/panel filtering, missing targets: {missing_targets2}")

keep_cols = sorted(set(tfs_in_data) | set(targets))
expr_sub = expr.loc[:, keep_cols].copy()

print("Scanpy matrix (cells x genes):", expr.shape)
print("Final (TFs+targets):", expr_sub.shape)

out_expr = "/content/expr_sub_for_grnboost2.tsv"
expr_sub.to_csv(out_expr, sep="\t")
print("Saved:", out_expr)


TFs found in dataset after HVG/panel filtering: 194
Scanpy matrix (cells x genes): (6033, 3003)
Final (TFs+targets): (6033, 196)
Saved: /content/expr_sub_for_grnboost2.tsv


In [ ]:

targets = ["PRM1", "PRM2"]
must_keep_tfs = ["CREM", "TBPL1", "HMGB4", "RPS4X"]
TOP_N_TFS = 500


tfs_in_panel = [t for t in tfs_in_data if t in X.columns]


tf_var = X[tfs_in_panel].var(axis=0).sort_values(ascending=False)

top_tfs = set(tf_var.head(TOP_N_TFS).index)

top_tfs |= set([t for t in must_keep_tfs if t in X.columns])

# Final expression matrix for GRNBoost2
keep_cols = sorted(top_tfs | set(targets))
expr_sub = X.loc[:, keep_cols].copy()

print("TFs kept:", len(top_tfs))
print("Final matrix (cells x genes):", expr_sub.shape)

out_expr = "/content/expr_sub_for_grnboost2.tsv"
expr_sub.to_csv(out_expr, sep="\t")
print("Saved:", out_expr)


TFs kept: 194
Final matrix (cells x genes): (6033, 196)
Saved: /content/expr_sub_for_grnboost2.tsv


In [ ]:
%%bash
set -e

./bin/micromamba run -n scenic python - << 'PY'
import pandas as pd
from arboreto.algo import grnboost2
from arboreto.utils import load_tf_names
from distributed import Client, LocalCluster

expr_path = "/content/expr_sub_for_grnboost2.tsv"
tf_path = "/content/allTFs_hg38.txt"
TARGETS = ["PRM1", "PRM2"]

# Load expression matrix (cells x genes)
ex_prm = pd.read_csv(expr_path, sep="\t", index_col=0)

# TF list restricted to genes present (targets excluded)
tf_names = load_tf_names(tf_path)
tf_names_filt = [tf for tf in tf_names if tf in ex_prm.columns and tf not in TARGETS]

print("Loaded expr:", ex_prm.shape)
print("TFs used:", len(tf_names_filt))

cluster = LocalCluster(
    processes=False,
    threads_per_worker=2,
    n_workers=4,
    dashboard_address=None
)
client = Client(cluster)

network = grnboost2(
    expression_data=ex_prm,
    tf_names=tf_names_filt,
    client_or_address=client
)

# Keep only edges into PRM1 / PRM2
sub = (
    network[network["target"].isin(TARGETS)]
    .sort_values(["target", "importance"], ascending=[True, False])
)

network.to_csv("/content/grnboost2_full.tsv", sep="\t", index=False)
sub.to_csv("/content/grnboost2_PRM1_PRM2_regulators.tsv", sep="\t", index=False)

print("Done.")
print("All edges:", len(network))
print("Edges into PRM1/PRM2:", len(sub))

client.close()
cluster.close()
PY


Loaded expr: (6033, 196)
TFs used: 194
Done.
All edges: 13649
Edges into PRM1/PRM2: 245


In [ ]:
import pandas as pd

in_path = "/content/grnboost2_PRM1_PRM2_regulators.tsv"
targets = ["PRM1", "PRM2"]
inspect_tfs = ["CREM", "TBPL1", "HMGB4", "RPS4X"]

df = pd.read_csv(in_path, sep="\t")

for tgt in targets:
    print("\n" + "="*60)
    print(f"Top 30 TFs regulating {tgt}")
    print("="*60)

    sub = (
        df[df["target"] == tgt]
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    top30 = sub.head(30).copy()
    top30["rank"] = top30.index + 1
    print(top30[["rank", "TF", "importance"]].to_string(index=False))

    print("\n-- Positions of TFs of interest --")
    for tf in inspect_tfs:
        hit = sub[sub["TF"] == tf]
        if hit.empty:
            print(f"{tf}: NOT FOUND")
        else:
            rank = hit.index[0] + 1
            imp = hit.iloc[0]["importance"]
            print(f"{tf}: rank={rank}, importance={imp:.4f}, in top30={'YES' if rank<=30 else 'NO'}")



Top 30 TFs regulating PRM1
 rank      TF  importance
    1   HMGB4   28.294336
    2   HMGB1   16.016692
    3   H2AFZ   15.354370
    4  ZNF683   11.757259
    5    SMC3    8.667127
    6   RPS4X    8.199282
    7   HSPA5    7.797207
    8     RAN    5.581839
    9    ILF2    4.364721
   10   HMGB2    4.157193
   11     JUN    3.383145
   12   TBPL1    2.745419
   13   CEBPD    2.692975
   14    CD59    2.603589
   15    YBX1    2.408357
   16  GTF2A2    2.332658
   17   TFDP2    1.939562
   18     UBB    1.938409
   19   NR2F2    1.734173
   20  FIP1L1    1.629929
   21 ZNF518A    1.348546
   22   ANXA1    1.304241
   23    CREM    1.200570
   24   STAT4    1.003518
   25    FHL2    0.880064
   26  CTNNB1    0.840145
   27    JUNB    0.757118
   28  ZBTB20    0.757043
   29   TIGD4    0.728319
   30    GOT1    0.715214

-- Positions of TFs of interest --
CREM: rank=23, importance=1.2006, in top30=YES
TBPL1: rank=12, importance=2.7454, in top30=YES
HMGB4: rank=1, importance=28.2943, 